# G2Gs 数据处理

这一份 notebook 负责回答一个非常重要的问题：

**模型看到的“数据”，到底是怎样从原始反应一步步变出来的？**

很多同学在第一次学习逆合成模型时，往往更容易把注意力放在网络结构上，但其实数据处理本身已经包含了大量关键的化学信息。G2Gs 能够顺利工作，很大程度上正是依赖这里的预处理规则。

## 学完这一份后，你应该能做到

- 看懂一条反应 SMILES 是怎样拆成 reactant 和 product 的。
- 理解 atom map 在反应数据中的作用。
- 明白 reaction center 是怎样从反应差分中定义出来的。
- 理解 synthons 是怎样从 product 断键得到的。
- 知道为什么 G2Gs 会被设计成“两阶段”模型。

## 先看一遍完整流程

```text
原始反应 CSV
    ↓
读取反应 SMILES
    ↓
RDKit 分子对象（reactant / product）
    ↓
分子图表示（node_feature / edge_feature / atom_map）
    ↓
反应差分分析
    ↓
reaction center 标注
    ↓
由 product 生成 synthons
    ↓
构造两类教学数据集
    ↓
导出给后续模型 notebook 使用
```

建议你边运行边问自己两个问题：

1. 这一格的输入是什么？
2. 这一格的输出会在后面哪一步继续使用？


In [ ]:
import os
import sys
from pathlib import Path


def find_project_root(start=None):
    here = Path(start or os.getcwd()).resolve()
    if here.is_file():
        here = here.parent
    for candidate in [here, *here.parents]:
        if (candidate / "teaching_demos").exists() and (candidate / "source_repos").exists():
            return candidate
    raise FileNotFoundError("无法定位项目根目录")


PROJECT_ROOT = find_project_root()
TUTORIAL_DIR = PROJECT_ROOT / "teaching_demos/2.single_step_retro_tutorial/2.3.semi-template/2.3.1.g2gs"
CODE_DIR = TUTORIAL_DIR / "code"
DATA_DIR = TUTORIAL_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

print(f"项目根目录: {PROJECT_ROOT}")
print(f"教程目录: {TUTORIAL_DIR}")
print(f"代码目录: {CODE_DIR}")
print(f"数据目录: {DATA_DIR}")

import pandas as pd
from IPython.display import display
from rdkit import Chem
from g2gs_tutorial import (
    build_center_identification_dataset,
    build_synthon_completion_dataset,
    compute_reaction_difference,
    demo_csv_path,
    draw_reaction_pair_image,
    export_processed_demo,
    extract_synthons,
    identify_reaction_center,
    load_demo_reactions,
    molecule_to_graph_tensor,
    source_mapping_frame,
)


## 1. 先建立源码对照关系

在正式处理数据之前，我们先把“教学代码中的函数”和“`torchdrug` 源码中的原始实现”对应起来。

这样做的好处是：

- 你在 notebook 里看到的每一步都不是凭空设计的。
- 如果你以后想回去读原始源码，可以知道应该去哪个文件、哪个函数继续深挖。


In [ ]:
display(source_mapping_frame())

## 2. 读取微型教学数据集

真实的 `USPTO50k` 数据集很大，不适合第一次学习时直接逐条展开分析。

因此这里选了 5 条规模适中、结构比较清晰的反应作为课堂示例。它们覆盖了几个常见场景：

- 新成键反应
- 官能团修饰
- 保护基引入
- 酯化
- 更复杂的原子中心变化

学习时不需要急着记住每条反应的化学背景，先把注意力放在“数据是怎样被模型表示”的过程上。


In [ ]:
demo_frame = pd.read_csv(demo_csv_path())
display(demo_frame[["reaction_id", "class", "reaction_name"]])

examples = load_demo_reactions()
print(f"共载入 {len(examples)} 条反应")


## 3. 先看一条原始反应

在进入图表示之前，先直观看一条 `(reactant, product)` 对。

这一步的目的不是做机理分析，而是帮助你建立最基本的观察习惯：

- 哪些原子在 reactant 和 product 中是同一个？
- 哪些键发生了变化？
- 为什么程序后面需要 atom map 来对齐两个分子？


In [ ]:
example = examples[0]
print("reaction_id:", example.reaction_id)
print("reaction_name:", example.reaction_name)
print("rxn_smiles:", example.rxn_smiles)
display(draw_reaction_pair_image(example.reactant_mol, example.product_mol, ["Reactant", "Product"]))


## 4. 分子图编码

G2Gs 并不会直接读取一串 SMILES 字符，而是先把分子转成图结构。

在图里：

- **原子** 是节点
- **化学键** 是边
- 每个节点和边都带有自己的特征向量

这里请重点关注三个字段：

- `node_feature`：描述原子的类型和局部性质
- `edge_feature`：描述键的类型和相关属性
- `atom_map`：告诉我们这个原子在反应前后如何对应

其中最关键的是 `atom_map`。后面所有“找差异”“找中心”“切 synthons”的步骤，都依赖它。


In [ ]:
product_graph = molecule_to_graph_tensor(example.product_mol, atom_feature_mode="center_identification")

graph_summary = pd.DataFrame(
    [
        {"field": "num_node", "value": product_graph.num_node},
        {"field": "num_edge (directed)", "value": product_graph.num_edge},
        {"field": "node_feature_dim", "value": product_graph.node_feature.shape[-1]},
        {"field": "edge_feature_dim", "value": product_graph.edge_feature.shape[-1]},
    ]
)
display(graph_summary)
display(
    pd.DataFrame(
        {
            "atom_index": list(range(product_graph.num_node)),
            "atom_map": product_graph.atom_map.tolist(),
        }
    )
)


## 5. 反应差分与 reaction center 标注

这一节是整份数据处理里最关键的部分之一。

程序并不是“凭感觉”决定哪里是反应中心，而是通过比较 reactant 和 product 的图结构差异，按规则一步步判断：

1. product 中是否出现了 reactant 里没有的新键？
2. 是否有键类型发生变化？
3. 是否只有单个原子的局部性质发生变化？

你可以把 reaction center 理解成：**模型在第一阶段最需要优先关注的位置**。


In [ ]:
rows = []
for item in examples:
    center = identify_reaction_center(item.reactant_mol, item.product_mol)
    diff = center["difference"]
    rows.append(
        {
            "reaction_id": item.reaction_id,
            "reaction_name": item.reaction_name,
            "center_type": center["center_type"],
            "reaction_center": center["reaction_center"],
            "added_bonds": diff["added_bonds"],
            "modified_bonds": diff["modified_bonds"],
            "hydrogen_changed_atoms": diff["hydrogen_changed_atoms"],
            "valid": center["valid"],
        }
    )
display(pd.DataFrame(rows))


## 6. 断键得到 synthons

当程序找到了 reaction center 之后，下一步并不是立刻输出 reactants，而是先把 product 切成 synthons。

可以把 synthons 理解成一种“中间态片段”：

- 它们比完整 product 更接近目标 reactants。
- 它们把原本复杂的逆合成任务拆成了更小的子问题。

这也是 G2Gs 采用两阶段设计的原因之一：

- 第一阶段先解决“从哪里断开”
- 第二阶段再解决“怎样把片段补成 reactants”


In [ ]:
example = next(item for item in examples if item.reaction_id == "demo_01")
center = identify_reaction_center(example.reactant_mol, example.product_mol)
pairs = extract_synthons(example.reactant_mol, example.product_mol, center)

print("reaction_center:", center["reaction_center"])
print("生成的 (reactant, synthon) 对数量:", len(pairs))
for pair_id, (reactant_mol, synthon_mol) in enumerate(pairs):
    print(f"pair {pair_id}:")
    print("  reactant:", Chem.MolToSmiles(reactant_mol, canonical=True))
    print("  synthon :", Chem.MolToSmiles(synthon_mol, canonical=True))
    display(draw_reaction_pair_image(reactant_mol, synthon_mol, ["Reactant", "Synthon"]))


## 7. 导出教学版数据集

最后，我们把处理好的结果导出成两份小型数据文件：

- `center_identification_demo.pkl`
- `synthon_completion_demo.pkl`

你可以把它们理解成后续模型 notebook 的“输入材料”。

如果这一节能顺利运行，说明你已经完成了从原始反应到模型训练样本的完整预处理流程。


In [ ]:
center_dataset = build_center_identification_dataset(examples)
synthon_dataset = build_synthon_completion_dataset(examples)
saved = export_processed_demo(center_dataset, synthon_dataset, PROCESSED_DIR)

print("center samples:", len(center_dataset))
print("synthon samples:", len(synthon_dataset))
print(saved)


In [ ]:
center_sample = center_dataset[0]
synthon_sample = synthon_dataset[0]

display(
    pd.DataFrame(
        [
            {
                "dataset": "center_identification",
                "reaction_id": center_sample["reaction_id"],
                "product_nodes": center_sample["product_graph"].num_node,
                "product_edges": center_sample["product_graph"].num_edge,
                "center_target": center_sample["center_target"]["reaction_center"],
            },
            {
                "dataset": "synthon_completion",
                "reaction_id": synthon_sample["reaction_id"],
                "synthon_nodes": synthon_sample["synthon_graph"].num_node,
                "synthon_edges": synthon_sample["synthon_graph"].num_edge,
                "oracle_action_count": len(synthon_sample["oracle_actions"]),
            },
        ]
    )
)
